# E2 -- Shots finitos + ruido NISQ

**Objetivo**: el régimen más realista de los cuatro (E0-E2): `WINNER_MODEL`
(y `cong`, si `INCLUDE_CONG_IN_E1_E2`) bajo 1024 shots fijos + transpilación
a un mapa de acoplamiento fijo de 16 qubits + una de las 7 condiciones de
ruido (`qcnn_benchmark.noise.NOISE_CONDITIONS`: ninguna, 3 aisladas, 3
compuestas). La variable independiente es la **condición de ruido**, no el
modelo.

Wei et al. se trata aparte (`noise.make_wei_noisy_predict_proba`, sin
transpilación): su lectura es un `qml.expval(qml.Hamiltonian(...))`, que
`qml.transforms.transpile` no soporta -- ver `src/qcnn_benchmark/execution/
transpile.py`.

**Reutiliza sin cambios** `train_binary_classifier` y el mismo criterio
estadístico de E0A/E0B/E1 -- solo cambia `predict_proba_fn` (viene de
`qcnn_benchmark.noise`).

**Antes de correr la matriz completa, lee la Sec. 2** -- el costo de este
régimen NO escala linealmente desde E0/E1.


In [ ]:
import json
import pathlib
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pennylane as qml

from qcnn_benchmark.config import load_config
from qcnn_benchmark.data import load_mnist_pool, load_fashion_mnist_pool
from qcnn_benchmark.representations import build_pca_dataset, build_amplitude_dataset
from qcnn_benchmark.models import qcnn_hur, qcnn_wei, qcnn_gong, qcnn_cong
from qcnn_benchmark.training import train_binary_classifier, uniform_pi_init, normal_init
from qcnn_benchmark.execution import COUPLING_MAP_16Q
from qcnn_benchmark.noise import make_noisy_predict_proba, make_wei_noisy_predict_proba, NOISE_CONDITIONS
from qcnn_benchmark.execution.shots import n_trainable_params
from qcnn_benchmark.metrics import batch_accuracy, confusion_counts
from qcnn_benchmark.stats import (
    bootstrap_bca_ci,
    paired_sign_permutation_test,
    holm_correction,
    cohens_dz,
    probability_of_superiority,
)


## 0. INPUTS MANUALES -- completar antes de correr

Mismos valores que en `E1_finite_shots.ipynb` (reutiliza `WINNER_MODEL` de
`E0A_seleccion_candidato.ipynb` e `INCLUDE_CONG_IN_E1_E2` decidido tras
`E0B_validez_aportacion.ipynb`).


In [ ]:
WINNER_MODEL = None          # <-- de E0A_seleccion_candidato.ipynb
INCLUDE_CONG_IN_E1_E2 = None # <-- True/False, decidido tras ver E0B (mismo valor que en E1_finite_shots.ipynb)

assert WINNER_MODEL in ("hur", "wei", "gong"), "Completa WINNER_MODEL arriba"
assert isinstance(INCLUDE_CONG_IN_E1_E2, bool), "Completa INCLUDE_CONG_IN_E1_E2 arriba (True o False)"

MODELS_TO_RUN = [WINNER_MODEL] + (["cong"] if INCLUDE_CONG_IN_E1_E2 else [])
print("MODELS_TO_RUN:", MODELS_TO_RUN)


## 1. Configuración declarativa

In [ ]:
FAST_SMOKE_TEST = False  # True = 2 semillas, pocas actualizaciones, 1 condicion de ruido -- solo para validar el pipeline

_CONFIG_CANDIDATES = [pathlib.Path("configs/e2.yaml"), pathlib.Path("../configs/e2.yaml")]
CONFIG_PATH = next(p for p in _CONFIG_CANDIDATES if p.exists())
config = load_config(CONFIG_PATH)

config = config.model_copy(update={"models": MODELS_TO_RUN})

if FAST_SMOKE_TEST:
    config = config.model_copy(update={
        "n_seeds": 2,
        "noise_conditions": ["none", "composite_high"],
        # batch_size tambien se reduce aqui (a diferencia de E0A/E0B/E1): con ruido,
        # el costo domina por EJEMPLO (matriz de densidad + parameter-shift/expansion
        # de Hamiltoniano), no por actualizacion -- dejar batch_size=25 harIa que incluso
        # el smoke test tardara minutos por corrida bajo las condiciones compuestas.
        "protocol": config.protocol.model_copy(update={"batch_size": 3, "n_updates": 5, "val_check_every": 5, "patience_checks": 2}),
    })

N_SHOTS_E2 = config.shots_budgets[0]  # fijo -- ver configs/e2.yaml

print("experiment_id:", config.experiment_id)
print("canonical_hash:", config.canonical_hash(), "(FAST_SMOKE_TEST)" if FAST_SMOKE_TEST else "")
print("datasets:", [d.name for d in config.datasets])
print("models:", config.models)
print("n_shots (fijo):", N_SHOTS_E2)
print("noise_conditions:", config.noise_conditions)
print("run_seeds:", config.run_seeds())


## 2. Estimación de costo de cómputo (MEDIDA, no escalada de E0/E1)

**Por qué no se puede solo escalar la referencia de E0A/E1**: la
simulación con canales de ruido corre sobre `default.mixed` (matriz de
densidad) en vez de `default.qubit` (estado puro) -- órdenes de magnitud
más lenta -- y el gradiente se calcula por regla de desplazamiento de
parámetro (∼2×n_params evaluaciones por ejemplo, cada una sobre una
matriz de densidad ya de por sí más cara). Esta celda **mide de verdad**
un gradiente de un solo ejemplo, bajo la condición de ruido más severa
declarada, para cada modelo en `MODELS_TO_RUN` -- puede tardar varios
minutos ella misma, eso ya es parte de la advertencia.


In [ ]:
_TOTAL_PARAMS = {"hur": qcnn_hur.TOTAL_PARAMS, "gong": qcnn_gong.TOTAL_PARAMS, "cong": qcnn_cong.TOTAL_PARAMS,
                  "wei": n_trainable_params("wei")}  # 37, no 46 -- ver execution/shots.py (beta fijo bajo shots/ruido)
_X_DIM = {"hur": 16, "gong": 8, "cong": 8}  # wei se arma aparte (amplitud, 1024)

_rng = np.random.default_rng(0)
_worst_condition = "composite_high" if "composite_high" in config.noise_conditions else config.noise_conditions[-1]

print(f"Midiendo el costo de UNA actualizacion real (batch={config.protocol.batch_size}) bajo")
print(f"'{_worst_condition}', {N_SHOTS_E2} shots (puede tardar minutos -- para modelos con")
print("encoding de amplitud como Wei, StatePrep se decompone en cientos de puertas via")
print("Mottonen, y la condicion 'composite_high' inserta 2 canales de ruido por cada una,")
print("asi que el costo real puede superar bastante una extrapolacion desde un solo ejemplo).\n")

per_model_seconds_per_update = {}
for model_name in config.models:
    n_params = _TOTAL_PARAMS[model_name]
    if model_name == "wei":
        rep_tmp = build_amplitude_dataset(*load_mnist_pool(normalize=False), class_pos=1, class_neg=0, verbose=False)
        pp = make_wei_noisy_predict_proba(N_SHOTS_E2, _worst_condition)
        init_fn_tmp = normal_init
    else:
        rep_tmp = build_pca_dataset(*load_mnist_pool(), class_pos=1, class_neg=0, n_components=_X_DIM[model_name], verbose=False)
        pp = make_noisy_predict_proba(model_name, N_SHOTS_E2, _worst_condition, coupling_map=COUPLING_MAP_16Q)
        init_fn_tmp = uniform_pi_init

    t0 = time.time()
    _ = train_binary_classifier(
        pp, n_params, rep_tmp, init_fn_tmp,
        run_seed=0, batch_size=config.protocol.batch_size, n_updates=1,
        val_check_every=999, patience_checks=999, verbose=False, tag="cost-probe",
    )
    dt = time.time() - t0
    per_model_seconds_per_update[model_name] = dt
    print(f"  {model_name}: {dt:.1f} s / actualizacion real (batch={config.protocol.batch_size}, {n_params} parametros)")

total_seconds = sum(
    per_model_seconds_per_update[m] * config.protocol.n_updates
    * len(config.noise_conditions) * config.n_seeds * len(config.datasets)
    for m in config.models
)
print()
print(f"Estimacion (extrapolacion LINEAL desde 1 actualizacion real, condicion mas severa, protocolo declarado):")
print(f"  ~{total_seconds/3600:.1f} horas (~{total_seconds/86400:.1f} dias)")
print()
print("Si ese numero es inviable, antes de lanzar la matriz completa considera:")
print("  - Reducir protocol.batch_size y/o protocol.n_updates SOLO para E2, p.ej.:")
print("      config = config.model_copy(update={\'protocol\': config.protocol.model_copy(")
print("          update={\'batch_size\': 5, \'n_updates\': 50})})")
print("  - Reducir n_seeds y/o el numero de condiciones de ruido en config.noise_conditions")
print("  - Nota: cualquiera de estos cambios modifica canonical_hash() -- es intencional,")
print("    documenta que corriste una version reducida del protocolo de E2, no la nominal.")


## 3. Registro de modelos y representaciones

In [ ]:
MODEL_REGISTRY = {
    "hur": {"module": qcnn_hur, "init": uniform_pi_init, "normalize": True,
            "build_rep": lambda x, y, pos, neg: build_pca_dataset(x, y, pos, neg, n_components=16, verbose=False)},
    "wei": {"module": qcnn_wei, "init": normal_init, "normalize": False,
            "build_rep": lambda x, y, pos, neg: build_amplitude_dataset(x, y, pos, neg, verbose=False)},
    "gong": {"module": qcnn_gong, "init": uniform_pi_init, "normalize": True,
             "build_rep": lambda x, y, pos, neg: build_pca_dataset(x, y, pos, neg, n_components=8, verbose=False)},
    "cong": {"module": qcnn_cong, "init": uniform_pi_init, "normalize": True,
             "build_rep": lambda x, y, pos, neg: build_pca_dataset(x, y, pos, neg, n_components=8, verbose=False)},
}

_POOL_CACHE = {}


def get_pool(source, normalize):
    key = (source, normalize)
    if key not in _POOL_CACHE:
        loader = load_mnist_pool if source == "mnist" else load_fashion_mnist_pool
        _POOL_CACHE[key] = loader(normalize=normalize)
    return _POOL_CACHE[key]


def make_predict_proba(model_name, condition):
    """predict_proba_fn bajo N_SHOTS_E2 + `condition` + (transpilacion a
    COUPLING_MAP_16Q, salvo para wei -- ver docstring del notebook)."""
    if model_name == "wei":
        return make_wei_noisy_predict_proba(N_SHOTS_E2, condition)
    return make_noisy_predict_proba(model_name, N_SHOTS_E2, condition, coupling_map=COUPLING_MAP_16Q)


## 4. Ejecución (matriz: datasets × modelos × condiciones de ruido × semillas)

In [ ]:
RESULTS_DIR = pathlib.Path("results") if pathlib.Path("results").exists() else pathlib.Path("../results")
raw_path = RESULTS_DIR / f"e2_raw_{config.canonical_hash()}.csv"

rows = []
t_start = time.time()
for dataset in config.datasets:
    for model_name in config.models:
        entry = MODEL_REGISTRY[model_name]
        x_all, y_all = get_pool(dataset.source, entry["normalize"])
        rep = entry["build_rep"](x_all, y_all, dataset.class_pos, dataset.class_neg)
        n_params = n_trainable_params(model_name)  # != entry["module"].TOTAL_PARAMS solo para "wei"

        for condition in config.noise_conditions:
            predict_proba = make_predict_proba(model_name, condition)

            for seed in config.run_seeds():
                tag = f"{dataset.name}-{model_name}-{condition}-seed{seed}"
                result = train_binary_classifier(
                    predict_proba, n_params, rep, entry["init"],
                    run_seed=seed,
                    batch_size=config.protocol.batch_size,
                    n_updates=config.protocol.n_updates,
                    learning_rate=config.protocol.learning_rate,
                    beta1=config.protocol.beta1,
                    beta2=config.protocol.beta2,
                    clip_norm=config.protocol.clip_norm,
                    val_check_every=config.protocol.val_check_every,
                    patience_checks=config.protocol.patience_checks,
                    min_delta=config.protocol.min_delta,
                    verbose=False,
                    tag=tag,
                )
                train_acc = batch_accuracy(predict_proba, result["params"], rep["X_train"], rep["y_train"])
                test_acc = batch_accuracy(predict_proba, result["params"], rep["X_test"], rep["y_test"])
                # E2 es la primera ejecución forward-only de las métricas predictivas
                # completas (ver metrics/__init__.py): tp/fp/fn/tn cuestan $0 extra
                # (predict_labels ya se recorre para test_acc) y bastan para derivar
                # precision/recall/F1/balanced accuracy/macro-F1 post-hoc, sin guardar
                # predicciones por muestra.
                test_confusion = confusion_counts(predict_proba, result["params"], rep["X_test"], rep["y_test"])
                rows.append({
                    "dataset": dataset.name, "model": model_name, "noise_condition": condition, "seed": seed,
                    "train_acc": train_acc, "test_acc": test_acc,
                    "test_tp": test_confusion["tp"], "test_fp": test_confusion["fp"],
                    "test_fn": test_confusion["fn"], "test_tn": test_confusion["tn"],
                    "n_updates_run": result["n_updates_run"], "stopped_early_at": result["stopped_early_at"],
                    "best_val_loss": result["best_val_loss"],
                })
                print(f"[{tag}] test_acc={test_acc:.4f}  (t={time.time()-t_start:.0f}s acumulado)")

                raw_df = pd.DataFrame(rows)
                raw_df.to_csv(raw_path, index=False)

print("Total:", time.time() - t_start, "s")
raw_df = pd.DataFrame(rows)
raw_df


## 5. Resultados crudos

In [ ]:
raw_df = pd.read_csv(raw_path)
display(raw_df)
raw_df.pivot_table(index=["dataset", "model"], columns="noise_condition", values="test_acc", aggfunc=list)


## 6. Métricas agregadas (media ± IC bootstrap BCa por dataset, modelo y condición de ruido)

In [ ]:
agg_rows = []
for (dataset_name, model_name, condition), group in raw_df.groupby(["dataset", "model", "noise_condition"]):
    accs = group["test_acc"].to_numpy()
    lo, hi = bootstrap_bca_ci(
        accs, n_bootstrap=config.statistical_criterion.n_bootstrap,
        confidence_level=config.statistical_criterion.confidence_level,
        rng=np.random.default_rng(0),
    )
    agg_rows.append({
        "dataset": dataset_name, "model": model_name, "noise_condition": condition,
        "mean_test_acc": float(np.mean(accs)), "ci_lo": lo, "ci_hi": hi, "n_seeds": len(accs),
    })

agg_df = pd.DataFrame(agg_rows)
agg_df


## 7. Análisis estadístico (cada condición de ruido vs. "none", por modelo y dataset)

Compara cada condición de ruido contra la referencia sin ruido ("none")
dentro de cada (dataset, modelo) -- la pregunta relevante aquí es cuánto
degrada cada condición respecto al régimen limpio, no las condiciones
entre sí.


In [ ]:
comparison_rows = []
for (dataset_name, model_name), group in raw_df.groupby(["dataset", "model"]):
    pivot = group.pivot(index="seed", columns="noise_condition", values="test_acc")
    if "none" not in pivot.columns:
        continue
    baseline = pivot["none"].to_numpy()
    for condition in config.noise_conditions:
        if condition == "none":
            continue
        y = pivot[condition].to_numpy()
        p_value, mean_diff = paired_sign_permutation_test(
            baseline, y, n_permutations=config.statistical_criterion.n_bootstrap,
            rng=np.random.default_rng(0),
        )
        comparison_rows.append({
            "dataset": dataset_name, "model": model_name, "condition": condition,
            "mean_diff_none_minus_condition": mean_diff, "p_value": p_value,
            "cohens_dz": cohens_dz(baseline, y), "prob_superiority_none_over_condition": probability_of_superiority(baseline, y),
        })

comparisons_df = pd.DataFrame(comparison_rows)
if len(comparisons_df):
    adjusted, rejected = holm_correction(comparisons_df["p_value"], alpha=config.statistical_criterion.alpha)
    comparisons_df["p_value_holm"] = adjusted
    comparisons_df["significant"] = rejected
comparisons_df


## 8. Figuras y tablas

In [ ]:
fig, axes = plt.subplots(1, len(config.datasets), figsize=(6 * len(config.datasets), 4.5), sharey=True)
if len(config.datasets) == 1:
    axes = [axes]
for ax, (dataset_name, group) in zip(axes, agg_df.groupby("dataset")):
    for model_name, model_group in group.groupby("model"):
        model_group = model_group.set_index("noise_condition").loc[config.noise_conditions]
        yerr = [model_group["mean_test_acc"] - model_group["ci_lo"], model_group["ci_hi"] - model_group["mean_test_acc"]]
        ax.errorbar(model_group.index, model_group["mean_test_acc"], yerr=yerr, marker="o", capsize=4, label=model_name)
    ax.set_title(dataset_name)
    ax.set_ylabel("Exactitud de prueba")
    ax.set_ylim(0.3, 1.0)
    ax.tick_params(axis="x", rotation=45)
    ax.grid(alpha=0.3, axis="y")
    ax.legend()
plt.suptitle("E2 -- media ± IC bootstrap BCa por condición de ruido (config hash: " + config.canonical_hash() + ")")
plt.tight_layout()
plt.show()


In [ ]:
if len(comparisons_df):
    display(comparisons_df.style.format({
        "mean_diff_none_minus_condition": "{:+.4f}", "p_value": "{:.4f}", "p_value_holm": "{:.4f}",
        "cohens_dz": "{:.2f}", "prob_superiority_none_over_condition": "{:.2f}",
    }))
else:
    print("Sin comparaciones (falta la condicion \'none\' en noise_conditions, o no hay otra condicion con la que compararla).")


## 9. Selección / conclusión -- ¿cuánto degrada cada condición de ruido?

In [ ]:
for (dataset_name, model_name), group in agg_df.groupby(["dataset", "model"]):
    group = group.set_index("noise_condition")
    if "none" not in group.index:
        continue
    baseline_acc = group.loc["none", "mean_test_acc"]
    print(f"{dataset_name} / {model_name} (none = {baseline_acc:.4f}):")
    for condition in config.noise_conditions:
        if condition == "none":
            continue
        acc = group.loc[condition, "mean_test_acc"]
        row = comparisons_df[
            (comparisons_df["dataset"] == dataset_name) & (comparisons_df["model"] == model_name)
            & (comparisons_df["condition"] == condition)
        ]
        sig = bool(row["significant"].iloc[0]) if len(row) else None
        print(f"  {condition:22s}: {acc:.4f}  (Δ={acc-baseline_acc:+.4f}, "
              f"{'significativo' if sig else 'NO significativo'} tras Holm)")


## 10. Registro de reproducibilidad

In [ ]:
import sklearn, scipy

provenance = {
    "experiment_id": config.experiment_id,
    "winner_model": WINNER_MODEL,
    "include_cong": INCLUDE_CONG_IN_E1_E2,
    "n_shots_fixed": N_SHOTS_E2,
    "canonical_hash": config.canonical_hash(),
    "config": json.loads(config.canonical_json()),
    "fast_smoke_test": FAST_SMOKE_TEST,
    "timestamp_utc": pd.Timestamp.utcnow().isoformat(),
    "raw_results_path": str(raw_path),
    "package_versions": {
        "pennylane": qml.__version__,
        "numpy": np.__version__,
        "scipy": scipy.__version__,
        "scikit_learn": sklearn.__version__,
        "pandas": pd.__version__,
    },
}

provenance_path = RESULTS_DIR / f"e2_provenance_{config.canonical_hash()}.json"
provenance_path.write_text(json.dumps(provenance, indent=2, ensure_ascii=False))
print("Provenance guardado en:", provenance_path)
print(json.dumps(provenance, indent=2, ensure_ascii=False))
